In [ ]:
import os
import json
import matplotlib.pyplot as plt
import geopandas as gpd
import pickle
from SCBIRL_Global_PE.utils import toWhoString  # 确保 utils 在路径中

# 加载城市底图
grid_path = "data/ss_city_grid/ss_city_grid_by_cover.shp"
city_grid = gpd.read_file(grid_path)

# 用户路径
base_dir = "data/user_data_migrt"
user_ids = sorted([uid for uid in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, uid))])

for who_str in user_ids:
    try:
        who = int(who_str)
        traj_file_path = os.path.join(base_dir, who_str, "all_traj.json")
        with open(traj_file_path, "r", encoding="utf-8") as f:
            stay_points_data = json.load(f)

        # 迭代起始时间
        with open('./data/user_data_migrt/' + f'{toWhoString(who)}/traveler_info.pkl', 'rb') as file:
            file = pickle.load(file)
        iter_start_date = file.iter_start_date

        # 分别收集迭代前和迭代后的点
        pre_lons, pre_lats = [], []
        post_lons, post_lats = [], []

        for sp in stay_points_data:
            for pt in sp["travel_chain"]:
                if sp["date"] < iter_start_date:
                    pre_lons.append(pt[0])
                    pre_lats.append(pt[1])
                else:
                    post_lons.append(pt[0])
                    post_lats.append(pt[1])

        # 画图
        fig, ax = plt.subplots(figsize=(8, 8))
        city_grid.plot(ax=ax, facecolor='white', edgecolor='lightgrey', linewidth=0.5)

        ax.scatter(pre_lons, pre_lats, c='blue', s=10, label='Pre-Iteration')
        ax.scatter(post_lons, post_lats, c='red', s=10, label='Post-Iteration')

        ax.set_title(f"User {who_str} - Stay Points (Before vs After Iteration)")
        ax.axis("equal")
        ax.legend()
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"❌ Failed to process {who_str}: {e}")